# 04 — Phase 2: MERT Embeddings + Clustering

**MERT-v1-330M** — trained on 160k hours of music with masked audio modeling.
No genre labels, no text alignment. Embeddings reflect perceptual acoustic content.

First run downloads ~1.3 GB from HuggingFace. Subsequent runs use local cache.

**Requires audio files**: personal MP3s in `../data/audio/personal/`  
and/or FMA small in `../data/audio/fma_small/`

⚠️ Embedding 8k FMA tracks takes ~2–3 hrs on MPS (CPU: ~6–8 hrs).
Start with personal library only to validate the approach first.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
from pathlib import Path

from anther_ml.embedding import load_mert, get_embedding, embed_batch
from anther_ml.cluster import fit_clusters, save_pipeline, cluster_summary
from anther_ml.similarity import SongIndex
from anther_ml.data import load_fma_tracks, get_audio_path

PERSONAL_DIR  = Path('../data/audio/personal')
FMA_AUDIO_DIR = Path('../data/audio/fma_small')
METADATA_DIR  = '../data/fma_metadata'

print('imports ok')

## Load MERT model

In [ ]:
model, processor, device = load_mert()
print(f'Running on: {device}')

## Embed personal library first

Fast (seconds per song). Good for early validation.

In [ ]:
personal_paths = sorted(PERSONAL_DIR.glob('*.mp3')) + sorted(PERSONAL_DIR.glob('*.wav'))
print(f'Found {len(personal_paths)} personal tracks')

if len(personal_paths) == 0:
    print('Add some MP3/WAV files to data/audio/personal/ first!')
else:
    personal_embeddings = embed_batch(model, processor, personal_paths, device, batch_size=4)
    print(f'Personal embeddings shape: {personal_embeddings.shape}')  # (N, 1024)
    
    personal_metadata = [
        {'name': p.stem, 'artist': 'personal', 'genre': 'unknown', 'source': 'personal'}
        for p in personal_paths
    ]
    
    np.save('../models/personal_embeddings.npy', personal_embeddings)
    with open('../models/personal_metadata.pkl', 'wb') as f:
        pickle.dump(personal_metadata, f)
    print('Saved personal embeddings.')

## Embed FMA small (optional — takes a while)

Skip this cell if you only have personal tracks for now.
Come back to it once you've downloaded fma_small audio.

In [ ]:
EMBED_FMA = False  # ← set True when fma_small audio is downloaded

if EMBED_FMA:
    tracks = load_fma_tracks(METADATA_DIR)
    small = tracks[tracks[('set', 'subset')] <= 'small']
    
    # Only embed tracks whose audio file exists
    fma_paths, fma_ids, fma_meta = [], [], []
    for tid in small.index:
        p = get_audio_path(FMA_AUDIO_DIR, tid)
        if p.exists():
            fma_paths.append(p)
            fma_ids.append(tid)
            fma_meta.append({
                'track_id': tid,
                'name': small.loc[tid, ('track', 'title')],
                'artist': small.loc[tid, ('artist', 'name')],
                'genre': small.loc[tid, ('track', 'genre_top')],
                'source': 'fma',
            })
    
    print(f'Found {len(fma_paths)} FMA audio files')
    print('Starting batch embedding (grab a coffee)...')
    
    fma_embeddings = embed_batch(model, processor, fma_paths, device, batch_size=8)
    np.save('../models/fma_embeddings.npy', fma_embeddings)
    with open('../models/fma_meta_phase2.pkl', 'wb') as f:
        pickle.dump(fma_meta, f)
    print(f'Done. Shape: {fma_embeddings.shape}')

## Combine and cluster

Cluster whatever you have — personal only, or personal + FMA.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load what's available
all_embeddings = []
all_metadata   = []

if Path('../models/personal_embeddings.npy').exists():
    emb = np.load('../models/personal_embeddings.npy')
    with open('../models/personal_metadata.pkl', 'rb') as f:
        meta = pickle.load(f)
    all_embeddings.append(emb)
    all_metadata.extend(meta)
    print(f'Personal: {emb.shape[0]} tracks')

if Path('../models/fma_embeddings.npy').exists():
    emb = np.load('../models/fma_embeddings.npy')
    with open('../models/fma_meta_phase2.pkl', 'rb') as f:
        meta = pickle.load(f)
    all_embeddings.append(emb)
    all_metadata.extend(meta)
    print(f'FMA: {emb.shape[0]} tracks')

X = np.vstack(all_embeddings)
print(f'\nTotal for clustering: {X.shape}')

In [ ]:
# With small datasets (personal only), reduce UMAP components and min_cluster_size
n = len(X)
umap_components = min(32, max(2, n // 10))
min_cluster = max(3, n // 20)

print(f'Clustering {n} tracks: UMAP({umap_components}d), HDBSCAN(min_cluster={min_cluster})')

labels, embedding_2d, scaler, pca, reducer, clusterer = fit_clusters(
    X,
    n_umap_components=umap_components,
    min_cluster_size=min_cluster,
)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct  = (labels == -1).mean() * 100
print(f'\nClusters: {n_clusters},  Noise: {noise_pct:.1f}%')

In [ ]:
# Visualize
genre_labels = [m.get('genre', 'unknown') or 'unknown' for m in all_metadata]
unique_genres = sorted(set(genre_labels))
palette = sns.color_palette('tab20', n_colors=len(unique_genres))
g2c = {g: palette[i] for i, g in enumerate(unique_genres)}
colors = [g2c[g] for g in genre_labels]

fig, ax = plt.subplots(figsize=(12, 9))
ax.scatter(embedding_2d[:, 0], embedding_2d[:, 1],
           c=colors, s=8 if n > 100 else 40, alpha=0.7, linewidths=0)

# Label personal songs by name
for i, m in enumerate(all_metadata):
    if m.get('source') == 'personal':
        ax.annotate(m['name'], (embedding_2d[i, 0], embedding_2d[i, 1]),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')

handles = [plt.Line2D([0],[0], marker='o', color='w',
                       markerfacecolor=g2c[g], markersize=7, label=g)
           for g in unique_genres]
ax.legend(handles=handles, title='Genre', bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=7)
ax.set_title('MERT embeddings — UMAP (Phase 2, genre-agnostic)')
plt.tight_layout()
plt.savefig('../models/umap_phase2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save pipeline and index for notebook 05
save_pipeline('../models/pipeline_phase2.pkl', scaler, reducer, clusterer, pca=pca)

for i, m in enumerate(all_metadata):
    m['cluster'] = int(labels[i])

index = SongIndex(X, all_metadata)
index.save('../models/index_phase2')

np.save('../models/embedding_2d_phase2.npy', embedding_2d)
np.save('../models/labels_phase2.npy', labels)

print('Saved pipeline, index, 2D embedding.')